In [9]:
import ee
import geemap

In [10]:
ee.Authenticate()
ee.Initialize(project="qflfw-511214")

In [11]:
# Define variables

lon_min = -1.23
lon_max = -0.89
lat_min = 44.72
lat_max = 44.96

collection = "COPERNICUS/S2_SR_HARMONIZED"
collection_prob = "COPERNICUS/S2_CLOUD_PROBABILITY"
bbox = ee.Geometry.BBox(lon_min, lat_min, lon_max, lat_max)
bands=["B4", "B3", "B2"]
visualization = {
    'min': 0.0,
    'max': 5000,
    'bands': bands,
}
start_date = ee.Date("2022-07-01")
end_date = ee.Date("2022-07-31")
cloudy_pixel_percent = 60
crs = "EPSG:32630"

In [12]:
def pull_dataset(start_date=start_date, end_date=end_date, cloud_percent=cloudy_pixel_percent, bbox=bbox):
    dataset = (
        ee.ImageCollection(collection)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', cloud_percent))
        .filterBounds(bbox)
    )

    return dataset

In [13]:
def check_crs(img, crs=crs, bands=bands):
    for band in bands:
        if (img.select(band).projection().getInfo()["crs"] != crs):
            raise Exception("The projection type does not match the desired one!")
        else:
            print(f"{band} OK!")

In [14]:
def display_map(dataset, bbox=bbox, vis_params=visualization, time_interval=0, zoom=9):
    m = geemap.Map()
    m.center_object(bbox, zoom=zoom)
    m.add_layer(dataset)
    if time_interval > 0:
        m.add_time_slider(dataset, vis_params=vis_params ,time_interval=time_interval)
    display(m)

In [15]:
def save_timelapse(dataset, start_date, end_date, bands=bands, dimensions=1080, output="output.gif", loop=0, frequency="day", fps=1):
    geemap.create_timelapse(
        collection=dataset,
        bands=bands,
        dimensions=dimensions,
        region=dataset.geometry(),
        start_date=start_date, 
        end_date=end_date, 
        out_gif=output,
        loop=loop,
        frequency=frequency,
        frames_per_second=fps
    )

In [18]:
# with cloud_pixel_percent = 70

dataset_jul_22 = pull_dataset(start_date, end_date)
check_crs(dataset_jul_22.first())
display_map(dataset_jul_22, time_interval=3)
save_timelapse(dataset_jul_22, start_date, end_date, output="fire.gif")

B4 OK!
B3 OK!
B2 OK!


Map(center=[44.83995872472239, -1.0600000000001695], controls=(WidgetControl(options=['position', 'transparent…

Total number of images: 5

ffmpeg is not installed on your computer.


Lower cloud percent allows fewer images into the collection.  
At cloud_pixel_percent = 70, we get 6 images  
At cloud_pixel_percent = 50, we get 5

In [17]:
# S2_Cloud_probability

def get_s2_sr_cld_col(start_date, end_date, aoi=bbox):
    s2_sr_col = (ee.ImageCollection(collection)
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', cloudy_pixel_percent)))

    # Import and filter s2cloudless.
    s2_cloudless_col = (ee.ImageCollection(collection_prob)
        .filterBounds(aoi)
        .filterDate(start_date, end_date))

    # Join the filtered s2cloudless collection to the SR collection by the 'system:index' property.
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))


### Yearly composites


In [29]:
# 2020, 2021, 2022, 2023, 2024. 
# Use six bands this time: B2 (blue), B3 (green), B4 (red), B8 (NIR), B11 (SWIR1) , B12 (SWIR2)

bands_6 = ["B2", "B3", "B4", "B8", "B11", "B12"]
dataset_median = dataset_jul_22.median().setDefaultProjection(crs=crs)

dataset_median.projection()